In [17]:
import re
import math

In [18]:
with open("temp.txt") as file:
    data = file.read()
    print(data)

Thy self thy foe, to thy sweet self too cruel:
Thou that art now the world's fresh ornament,
And only herald to the gaudy spring,
Within thine own bud buriest thy content,
And tender churl mak'st waste in niggarding:
Pity the world, or else this glutton be,
To eat the world's due, by the grave and thee.
When forty winters shall besiege thy brow,
And dig deep trenches in thy beauty's field,
Thy youth's proud livery so gazed on now,
Will be a tattered weed of small worth held:
Then being asked, where all thy beauty lies,
Where all the treasure of thy lusty days;
To say within thine own deep sunken eyes,
Were an all-eating shame, and thriftless praise.
How much more praise deserved thy beauty's use,
If thou couldst answer 'This fair child of mine
Shall sum my count, and make my old excuse'
Proving his beauty by succession thine.
This were to be new made when thou art old,
And see thy blood warm when thou feel'st it cold.
Look in thy glass and tell the face thou viewest,
Now is the time th

In [19]:
# Sentence Tokenizer
sentence_end = re.compile(r'([\.!?])\s+')
def tokenize_sentences(paragraph: str) -> list:
    # Normalize whitespace
    text = paragraph.strip().replace("\n", " ")
    
    # Split on '.', '!' or '?' plus following space
    parts = sentence_end.split(text)
    
    sentences = []
    for i in range(0, len(parts) - 1, 2):
        sent = parts[i] + parts[i+1]
        sentences.append(sent.strip())
    
    if len(parts) % 2 == 1 and parts[-1].strip():
        sentences.append(parts[-1].strip())
    
    return sentences

In [20]:
print((tokenize_sentences(data)))

["Thy self thy foe, to thy sweet self too cruel: Thou that art now the world's fresh ornament, And only herald to the gaudy spring, Within thine own bud buriest thy content, And tender churl mak'st waste in niggarding: Pity the world, or else this glutton be, To eat the world's due, by the grave and thee.", "When forty winters shall besiege thy brow, And dig deep trenches in thy beauty's field, Thy youth's proud livery so gazed on now, Will be a tattered weed of small worth held: Then being asked, where all thy beauty lies, Where all the treasure of thy lusty days; To say within thine own deep sunken eyes, Were an all-eating shame, and thriftless praise.", "How much more praise deserved thy beauty's use, If thou couldst answer 'This fair child of mine Shall sum my count, and make my old excuse' Proving his beauty by succession thine.", "This were to be new made when thou art old, And see thy blood warm when thou feel'st it cold.", 'Look in thy glass and tell the face thou viewest, Now 

In [21]:
def tokenize_sentence(paragraph: str):
    #protect urls, dates, emails
    URL_PATTERN = r"https?:\/\/(www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b([-a-zA-Z0-9()@:%_\+.~#?&//=]*)"
    EMAIL_PATTERN = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}(?:\.[a-zA-Z]{2,})?$'
    DATE_PATTERN   = r'\d{1,2}[-/]\d{1,2}[-/]\d{2,4}'

    protected = []
    def protect(match):
        protected.append(match.group(0))
        return f"__PROTECTED__{len(protected)-1}__"
    
    paragraph = re.sub(r'\.\.\.', protect, paragraph)
    paragraph = re.sub(URL_PATTERN, protect, paragraph)
    paragraph = re.sub(EMAIL_PATTERN, protect, paragraph)
    paragraph = re.sub(DATE_PATTERN, protect, paragraph)

    text = paragraph.strip().replace("\n", " ")

    sentence_end = re.compile(r"([\.?!])\s+")

    parts = sentence_end.split(text)

    sentences = []
    for i in range(0, len(parts)-1, 2):
        if i+1 < len(parts):
            sentence = parts[i] + parts[i+1]
        else:
            sentence = parts[i]

        for idx, item in enumerate(protected):
            sentence = sentence.replace(f"__PROTECTED__{idx}__", item)
        
        sentence = sentence.strip()
        if sentence:
            sentences.append(sentence)
        
    
    if len(parts) % 2 == 1 and parts[-1].strip():
        last = parts[-1].strip()

        for idx, item in enumerate(protected):
            last = last.replace(f"__PROTECTED__{idx}__", item)
        
        if last:
            sentences.append(last)
    
    return sentences


In [22]:
print(len((tokenize_sentence(data))))

15


In [23]:
import re

def tokenizer(sentence):
    URL_PATTERN = r"https?:\/\/(?:www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b(?:[-a-zA-Z0-9()@:%_\+.~#?&//=]*)"
    EMAIL_PATTERN = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}(?:\.[a-zA-Z]{2,})?'
    DATE_PATTERN = r'\d{1,2}[-/]\d{1,2}[-/]\d{2,4}'
    DECIMAL_PATTERN = r'\d+\.\d+'
    NUMBER_PATTERN = r'\d+'
    PUNCT_PATTERN = r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)*|[^\w\s]"

    token_re = re.compile(
        f"{URL_PATTERN}|{EMAIL_PATTERN}|{DATE_PATTERN}|{DECIMAL_PATTERN}|{NUMBER_PATTERN}|{PUNCT_PATTERN}"
    )

    return token_re.findall(sentence)


def word_tokenizer(sentences):
    tokenized_words = []
    for sent in sentences:
        tokens = tokenizer(sent)
        for token in tokens:
            if len(token) < 3:
                continue
            else:
                tokenized_words.append(token)
    
    return tokenized_words


In [24]:
print(len(word_tokenizer(tokenize_sentence(data))))

344


In [25]:
# n gram model. 
import string
def tokenize(text):
    for p in string.punctuation:
        text = text.replace(p, ' '+p+' ')
    t = text.split()
    return t

In [26]:
ngram_counter = {}
context = {}
n = 3
with open("temp.txt") as f:
    text = f.read()
    sentences = tokenize_sentence(text)

    for sentence in sentences:
        tokens = tokenizer(sentence)
        tokens = (n-1)*['<s>']+tokens
        
        for i in range(n-1, len(tokens)):
            context = tuple(tokens[i-p-1] for p in reversed(range(n-1)))
            target_word = tokens[i]
            print((context, target_word))

(('<s>', '<s>'), 'Thy')
(('<s>', 'Thy'), 'self')
(('Thy', 'self'), 'thy')
(('self', 'thy'), 'foe')
(('thy', 'foe'), ',')
(('foe', ','), 'to')
((',', 'to'), 'thy')
(('to', 'thy'), 'sweet')
(('thy', 'sweet'), 'self')
(('sweet', 'self'), 'too')
(('self', 'too'), 'cruel')
(('too', 'cruel'), ':')
(('cruel', ':'), 'Thou')
((':', 'Thou'), 'that')
(('Thou', 'that'), 'art')
(('that', 'art'), 'now')
(('art', 'now'), 'the')
(('now', 'the'), "world's")
(('the', "world's"), 'fresh')
(("world's", 'fresh'), 'ornament')
(('fresh', 'ornament'), ',')
(('ornament', ','), 'And')
((',', 'And'), 'only')
(('And', 'only'), 'herald')
(('only', 'herald'), 'to')
(('herald', 'to'), 'the')
(('to', 'the'), 'gaudy')
(('the', 'gaudy'), 'spring')
(('gaudy', 'spring'), ',')
(('spring', ','), 'Within')
((',', 'Within'), 'thine')
(('Within', 'thine'), 'own')
(('thine', 'own'), 'bud')
(('own', 'bud'), 'buriest')
(('bud', 'buriest'), 'thy')
(('buriest', 'thy'), 'content')
(('thy', 'content'), ',')
(('content', ','), 'And')

In [27]:
from collections import Counter
class NGramModel:
    def __init__(self, n=1):
        self.n = n
        self.ngram_counter = Counter()
        self.context_counter = Counter()
        self.vocab = set()
        self.vocab_size = 0

    def add_sentence_markers(self, sentence):
        return (self.n - 1) * ['<s>'] + sentence + ['</s>']

    def get_ngrams(self, sentence):
        marked_sentence = self.add_sentence_markers(sentence)
        ngrams = []
        for i in range(len(marked_sentence)- self.n + 1):
            ngram = tuple(marked_sentence[i: i+self.n])
            ngrams.append(ngram)
        return ngrams

    def train(self, sentences):
        for sentence in sentences:
            self.vocab.update(sentence)
        
        self.vocab.add('<s>')
        self.vocab.add('</s>')

        self.vocab_size = len(self.vocab)

        for sentence in sentences:
            ngrams = self.get_ngrams(sentence)
            
            for ngram in ngrams:
                self.ngram_counter[ngram] += 1
                if self.n > 1:
                    context = ngram[:-1]
                    self.context_counter[context] += 1
    
    def probability(self, ngram, smoothing='none', k=1):
        if isinstance(ngram, list):
            ngram = tuple(ngram)
        
        if self.n == 1:
            count = self.ngram_counter[ngram]
            total = sum(self.ngram_counter.values())
            if smoothing == 'add_one':
                return (count + 1) / (total + self.vocab_size)
            elif smoothing == 'add_k':
                return (count + k) / (total + k * self.vocab_size)
            elif smoothing == 'add_token_type':
                unique = len(self.ngram_counter)
                return (count + 1) / (total + unique)
            else:
                return count / total if total > 0 else 0
        else:
            context = ngram[:-1]
            count = self.ngram_counter[ngram]
            context_count = self.context_counter[context]
            if smoothing == 'add_one':
                return (count + 1) / (context_count + self.vocab_size)
            elif smoothing == 'add_k':
                return (count + k) / (context_count + k * self.vocab_size)
            elif smoothing == 'add_token_type':
                unique_types = len([ng for ng in self.ngram_counter.keys() if ng[:-1] == context])
                if unique_types == 0:
                    unique_types = 1
                return (count + 1) / (context_count + unique_types)
            else:
                return count / context_count if context_count > 0 else 0
    
    def sentence_log_probability(self, sentence, smoothing='none', k=1):
        """Compute log probability of a sentence as sum log P(ngram) over n-grams."""
        ngrams = self.get_ngrams(sentence)
        log_prob = 0.0
        for ngram in ngrams:
            p = self.probability(ngram, smoothing, k)
            if p > 0:
                log_prob += math.log(p)
            else:
                log_prob += float('-inf')  # Sentence impossible
                break
        return log_prob

In [28]:
with open("msd.txt") as f:
    text = f.read()
    sentences = tokenize_sentence(text)
    sentences = [tokenizer(sent) for sent in sentences]
    trigram_model = NGramModel(3)
    trigram_model.train(sentences)

In [29]:
with open("msd.txt") as f:
    text = f.read()
    sentences = tokenize_sentence(text)
    sentences = [tokenizer(sent) for sent in sentences]
    unigram_model = NGramModel(1)
    unigram_model.train(sentences)

In [30]:
import math
trigrams = list(trigram_model.ngram_counter.keys())

pmi_scores = []
for w1,w2,w3 in trigrams:
    c_w1w2w3 = trigram_model.ngram_counter.get((w1, w2, w3), 0)
    c_w1 = unigram_model.ngram_counter.get((w1, ), 0)
    c_w2 = unigram_model.ngram_counter.get((w2, ), 0)
    c_w3 = unigram_model.ngram_counter.get((w3, ), 0)

    if c_w1 == 0 or c_w2 == 0 or c_w3 == 0:
        pmi = None
    
    else:
        p_w1 = c_w1 / sum(unigram_model.ngram_counter.values())
        p_w2 = c_w2 / sum(unigram_model.ngram_counter.values())
        p_w3 = c_w3 / sum(unigram_model.ngram_counter.values())
        p_w1w2w3 = c_w1w2w3 / sum(trigram_model.ngram_counter.values())
        denom = p_w1 * p_w2 * p_w3
        if denom == 0 or p_w1w2w3 == 0:
            pmi = None
        else:
            pmi = math.log(p_w1w2w3 / denom)
        
    pmi_scores.append((w1, w2, w3, c_w1w2w3, c_w1, c_w2, c_w3, pmi))

print(pmi_scores)

[('<s>', '<s>', 'Mahendra', 1, 0, 0, 4, None), ('<s>', 'Mahendra', 'Singh', 1, 0, 4, 7, None), ('Mahendra', 'Singh', 'Dhoni', 4, 4, 7, 133, 10.464389730761773), ('Singh', 'Dhoni', 'is', 1, 7, 133, 15, 7.756339529659563), ('Dhoni', 'is', 'an', 3, 133, 15, 36, 7.2173430289268765), ('is', 'an', 'Indian', 1, 15, 36, 41, 7.295507801776212), ('an', 'Indian', 'professional', 1, 36, 41, 1, 10.003558002878423), ('Indian', 'professional', 'cricketer', 1, 41, 1, 5, 11.977639028900432), ('professional', 'cricketer', 'who', 1, 1, 5, 3, 14.59259880693663), ('cricketer', 'who', 'plays', 1, 5, 3, 4, 13.20630444581674), ('who', 'plays', 'as', 1, 3, 4, 38, 11.178156198524455), ('plays', 'as', 'a', 1, 4, 38, 80, 7.894741852518683), ('as', 'a', 'right', 1, 38, 80, 2, 8.587889033078628), ('a', 'right', '-', 2, 80, 2, 88, 8.441285558886753), ('right', '-', 'handed', 2, 2, 88, 4, 11.437017832440743), ('-', 'handed', 'batter', 1, 88, 4, 1, 11.437017832440743), ('handed', 'batter', 'and', 1, 4, 1, 126, 11.0780

In [31]:
pmi_scores2 = [row for row in pmi_scores if row[-1] is not None]
pmi_scores2.sort(key=lambda x: x[-1], reverse=True)
pmi_scores2[-10:]

[('and', 'was', 'the', 1, 126, 61, 333, 2.5453507469336074),
 ('series', 'in', 'the', 1, 38, 222, 333, 2.4522429764597318),
 ('Dhoni', '.', '</s>', 2, 133, 208, 204, 2.447788986831498),
 (',', 'in', 'a', 1, 170, 222, 80, 2.380146554442418),
 ('in', 'the', 'Indian', 1, 222, 333, 41, 2.37625706948181),
 ('India', 'in', 'the', 1, 46, 222, 333, 2.2611877396970224),
 ('and', 'in', 'the', 2, 126, 222, 333, 1.9466944097945846),
 (',', 'and', 'in', 1, 170, 126, 222, 1.9258912821648213),
 (',', 'and', 'the', 1, 170, 126, 333, 1.520426174056657),
 ('Dhoni', 'in', 'the', 1, 133, 222, 333, 1.1994800079643635)]

In [32]:
models = {
    'Unigram': NGramModel(1),
    'Bigram': NGramModel(2),
    'Trigram': NGramModel(3),
    'Quadrigram': NGramModel(4)
}

with open("msd.txt") as f:
    text = f.read()
    sentences = tokenize_sentence(text)
    sentences = [tokenizer(sent) for sent in sentences]

for name, model in models.items():
    model.train(sentences)
    print(f"Trained {name} model (vocab size: {model.vocab_size})")

Trained Unigram model (vocab size: 1196)
Trained Bigram model (vocab size: 1196)
Trained Trigram model (vocab size: 1196)
Trained Quadrigram model (vocab size: 1196)


In [33]:
smoothings = ['none', 'add_one', 'add_k']
total_sentences = len(sentences)
results = []
for name, model in models.items():
    for sm in smoothings:
        k = 1 if sm == 'add_k' else None  # Only for add_k
        total_logp = 0.0
        impossible_count = 0
        for sent in sentences:
            logp = model.sentence_log_probability(sent, smoothing=sm, k=k)
            if math.isinf(logp) and logp < 0:
                impossible_count += 1
            total_logp += logp
        avg_logp = total_logp / total_sentences if total_sentences > 0 else 0
        perplexity = math.exp(-avg_logp) if avg_logp > float('-inf') else float('inf')
        results.append({
            'Model': name,
            'Smoothing': sm,
            'Avg Log Prob': round(avg_logp, 4),
            'Perplexity': round(perplexity, 2) if not math.isinf(perplexity) else 'inf',
            'Impossible Sentences': impossible_count
        })

In [34]:
see = 'Dhoni finished the season with 283 runs in 5 matches.'
model.sentence_log_probability(tokenizer(see))

-5.318119993844216

In [134]:
print("\nEvaluation Results (on 1000 test sentences):")
print("| Model       | Smoothing   | Avg Log Prob | Perplexity | Impossible Sents |")
print("|-------------|-------------|--------------|------------|------------------|")
for res in results:
    print(f"| {res['Model']:<11} | {res['Smoothing']:<11} | {res['Avg Log Prob']:<12} | {res['Perplexity']:<10} | {res['Impossible Sentences']:<16} |")


Evaluation Results (on 1000 test sentences):
| Model       | Smoothing   | Avg Log Prob | Perplexity | Impossible Sents |
|-------------|-------------|--------------|------------|------------------|
| Unigram     | none        | -159.7738    | 2.4483629993920833e+69 | 0                |
| Unigram     | add_one     | -160.5704    | 5.430632456231198e+69 | 0                |
| Unigram     | add_k       | -160.5704    | 5.430632456231198e+69 | 0                |
| Bigram      | none        | -54.6855     | 5.618623634778318e+23 | 0                |
| Bigram      | add_one     | -159.7953    | 2.5017124672906744e+69 | 0                |
| Bigram      | add_k       | -159.7953    | 2.5017124672906744e+69 | 0                |
| Trigram     | none        | -18.1894     | 79352715.51 | 0                |
| Trigram     | add_one     | -172.828     | 1.1434832004425357e+75 | 0                |
| Trigram     | add_k       | -172.828     | 1.1434832004425357e+75 | 0                |
| Quadrigram 

In [2]:
from pathlib import Path
from collections import defaultdict, deque, Counter

INPUT_FILE = "msd.txt"
MAX_N = 4

def stream_tokens(filename):
    with open(filename, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            for tok in line.strip().split():
                t = tok.strip()
                if t:
                    yield t

def good_turing(counts, n, vocab_size):
    # Compute Nc: Nc[c] = number of n-grams with count c
    Nc = Counter(counts.values())
    N = sum(counts.values())
    N1 = Nc[1]
    if n == 1:
        U = len(counts)
        num_unseen = vocab_size - U
        P_unseen = N1 / (N * num_unseen) if num_unseen > 0 else 0.0
    else:
        num_unseen = vocab_size ** n - len(counts)
        Nc[0] = num_unseen
        P_unseen = N1 / (N * num_unseen) if num_unseen > 0 else 0.0
    probs = {}
    cstar_table = {}
    max_c = max(Nc) if Nc else 0
    for c in range(0, max_c + 1):
        Nc1 = Nc.get(c + 1, 0)
        Nc_c = Nc.get(c, 0)
        if c == 0:
            cstar = N1 / num_unseen if num_unseen > 0 else 0.0
        elif Nc1 > 0 and Nc_c > 0:
            cstar = (c + 1) * Nc1 / Nc_c
        else:
            cstar = c
        cstar_table[c] = (Nc_c, cstar)
    for gram, c in counts.items():
        Nc1 = Nc.get(c + 1, 0)
        if Nc1 > 0:
            c_star = (c + 1) * Nc1 / Nc[c]
            p = c_star / N
        else:
            p = c / N
        probs[gram] = p
    return probs, P_unseen, cstar_table

def write_gt(n: int, counts, probs, out_path: Path):
    rows = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    header = [f"w{i+1}" for i in range(n)] + ["count", "gt_prob"]
    with out_path.open("w", encoding="utf-8") as f:
        f.write("\t".join(header) + "\n")
        for gram, c in rows:
            p = probs[gram]
            f.write("\t".join(list(gram) + [str(c), f"{p:.8f}"]) + "\n")

AllCounts = defaultdict(lambda: defaultdict(int))
Vocab = set()
Window = deque(maxlen=MAX_N - 1)
OutputDir = Path("out")

# 1. Collect counts and build vocabulary
for token in stream_tokens(INPUT_FILE):
    Vocab.add(token)
    AllCounts[1][(token,)] += 1
    
    if MAX_N > 1:
        Hist = list(Window)
        for n_gram in range(2, MAX_N + 1):
            Need = n_gram - 1
            if len(Hist) >= Need:
                gram = tuple(Hist[-Need:] + [token])
                AllCounts[n_gram][gram] += 1
    Window.append(token)
    
VocabSize = len(Vocab)
print(f"Vocab size: {VocabSize}")

# 2. Compute GT probabilities and write output
NameMap = {1: "unigrams_gt.tsv", 2: "bigrams_gt.tsv", 3: "trigrams_gt.tsv", 4: "quadragrams_gt.tsv"}

for n_gram in range(1, MAX_N + 1):
    n_counts = AllCounts.get(n_gram)
    if not n_counts: continue

    Probs, P_unseen, CstarTable = good_turing(n_counts, n_gram, VocabSize)

    OutPath = OutputDir / NameMap.get(n_gram, f"{n_gram}grams_gt.tsv")
    write_gt(n_gram, n_counts, Probs, OutPath)

    print(f"\n--- {n_gram}-gram Model ---\nUnseen P: {P_unseen:.8g}\n{'C':>6} {'Nc':>8} {'C*':>12}")

    # C=0 row
    c0_Nc, c0_cstar = CstarTable[0]
    print(f"{0:6d} {c0_Nc:8d} {c0_cstar:12.4f}")

    # Top 99 C > 0
    freq_counts = sorted([(C, Nc) for C, (Nc, _) in CstarTable.items() if C > 0], key=lambda x: -x[1])[:99]
    for C, Nc in freq_counts:
        print(f"{C:6d} {Nc:8d} {CstarTable[C][1]:12.4f}")

Vocab size: 1387

--- 1-gram Model ---
Unseen P: 0
     C       Nc           C*
     0        0       0.0000
     1      901       0.4084
     2      184       1.5815
     3       97       1.7320
     4       42       3.4524
     6       30       3.9667
     5       29       6.2069
     7       17       5.1765
     8       11       4.9091
    12       10       7.8000
    10        8       8.2500
     9        6      13.3333
    11        6      20.0000
    13        6       2.3333
    18        3       6.3333
    15        2      16.0000
    16        2      16.0000
    20        2      20.0000
    24        2      24.0000
    34        2      17.5000
    38        2      38.0000
    41        2      41.0000
    14        1      30.0000
    19        1      40.0000
    22        1      23.0000
    23        1      48.0000
    26        1      26.0000
    28        1      29.0000
    29        1      29.0000
    31        1      31.0000
    35        1      36.0000
    36        1      

In [ ]:
from pathlib import Path
from collections import defaultdict, deque, Counter

TRAIN_FILENAME = "msd.txt"
MAX_N = 4
k = 1  # threshold for discounting (can be tuned)

def stream_tokens(path: Path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            for tok in line.strip().split():
                t = tok.strip()
                if t:
                    yield t

def count_ngrams(tokens, max_n=4):
    counts = {n: defaultdict(int) for n in range(1, max_n+1)}
    window = deque(maxlen=max_n-1)
    for tok in tokens:
        counts[1][(tok,)] += 1
        if max_n > 1:
            hist = list(window)
            hl = len(hist)
            for n in range(2, max_n+1):
                need = n - 1
                if hl >= need:
                    gram = tuple(hist[-need:] + [tok])
                    counts[n][gram] += 1
        window.append(tok)
    return counts

def mle_prob(gram, counts_n, counts_prev):
    # If unigram
    if len(gram) == 1:
        total = sum(counts_n.values())
        return counts_n.get(gram, 0) / total if total else 0.0
    # For n-gram (len>1), denominator is count(history)
    hist = gram[:-1]
    denom = counts_prev.get(hist, 0)
    return counts_n.get(gram, 0) / denom if denom else 0.0

def discount(c, Nc):
    """
    Discount formula:
      d_{r} = ((r + 1) * N_{r+1}) / (N_r * r)
    where Nc is a Counter mapping frequency r -> number of n-grams with frequency r
    """
    r = c
    if r > 0 and Nc.get(r, 0) > 0 and Nc.get(r+1, 0) > 0:
        return (r + 1) * Nc[r+1] / (Nc[r] * r)
    # fallback: no discount (or can use small epsilon)
    return 1.0

def get_Nc(counts_n):
    # counts_n are mapping gram -> count; we want Nc: r -> number of grams with freq r
    return Counter(counts_n.values())

def pkatz(w, h, counts, Nc_dict, k):
    """
    Top (4-gram) Katz probability P_katz(w | h) where h is a tuple of 3 words.
    If c > k -> discounted MLE
    Else -> back off with alpha(h) * lower-order probability
    """
    quad = h + (w,)
    c = counts[4].get(quad, 0)
    Nc = Nc_dict[4]
    if c > k:
        d = discount(c, Nc)
        return d * mle_prob(quad, counts[4], counts[3])
    else:
        alpha = compute_alpha(h, counts, Nc_dict, k, order=4)
        # lower order history for tri is last 2 words of h
        return alpha * pkatz_lower(w, h[1:], counts, Nc_dict, k)

def pkatz_lower(w, h, counts, Nc_dict, k):
    """
    Tri-gram level: h is tuple of 2 words (Wi-2, Wi-3 in your naming)
    If c > k -> discounted MLE on trigram
    Else -> backoff to bigram
    """
    tri = h + (w,)
    c = counts[3].get(tri, 0)
    Nc = Nc_dict[3]
    if c > k:
        d = discount(c, Nc)
        return d * mle_prob(tri, counts[3], counts[2])
    else:
        alpha = compute_alpha(h, counts, Nc_dict, k, order=3)
        return alpha * pkatz_lower2(w, h[1:], counts, Nc_dict, k)

def pkatz_lower2(w, h, counts, Nc_dict, k):
    """
    Bigram level: h is tuple of 1 word.
    If c > k -> discounted MLE on bigram
    Else -> backoff to unigram MLE
    """
    bi = h + (w,)
    c = counts[2].get(bi, 0)
    Nc = Nc_dict[2]
    if c > k:
        d = discount(c, Nc)
        return d * mle_prob(bi, counts[2], counts[1])
    else:
        alpha = compute_alpha(h, counts, Nc_dict, k, order=2)
        # unigram fallback
        return alpha * mle_prob((w,), counts[1], {})

def compute_alpha(h, counts, Nc_dict, k, order=4):
    """
    Compute alpha for history h of length order-1.
    Formula implemented:
      alpha(h) = (1 - sum_{c(w,h) > k} d_{w,h} * P_mle(w|h)) /
                 (1 - sum_{c(w,h) > 0} P_lower(w|h'))
    where P_lower is the backed-off lower-order probability for the corresponding w.
    We iterate over all vocabulary words, but only include terms in the sums when the
    higher-order count condition holds (c > k for numerator; c > 0 for denominator).
    """
    # numerator starts at 1, subtract discounted mass for words with c > k
    numer = 1.0
    denom = 1.0

    # iterate vocabulary (counts[1] keys are ('word',))
    for w_tuple in counts[1].keys():
        wtok = w_tuple[0]
        if order == 4:
            quad = h + (wtok,)
            c = counts[4].get(quad, 0)
            if c > k:
                d = discount(c, Nc_dict[4])
                numer -= d * mle_prob(quad, counts[4], counts[3])
        elif order == 3:
            tri = h + (wtok,)
            c = counts[3].get(tri, 0)
            if c > k:
                d = discount(c, Nc_dict[3])
                numer -= d * mle_prob(tri, counts[3], counts[2])
        elif order == 2:
            bi = h + (wtok,)
            c = counts[2].get(bi, 0)
            if c > k:
                d = discount(c, Nc_dict[2])
                numer -= d * mle_prob(bi, counts[2], counts[1])

    # Denominator: subtract lower-order (backed-off) probabilities
    # but only for words where the higher-order count c > 0 (i.e., observed with this history)
    for w_tuple in counts[1].keys():
        wtok = w_tuple[0]
        if order == 4:
            quad = h + (wtok,)
            c = counts[4].get(quad, 0)
            if c > 0:
                # lower order for quad is trigram history h[1:]
                denom -= pkatz_lower(wtok, h[1:], counts, Nc_dict, k)
        elif order == 3:
            tri = h + (wtok,)
            c = counts[3].get(tri, 0)
            if c > 0:
                # lower order for tri is bigram history h[1:]
                denom -= pkatz_lower2(wtok, h[1:], counts, Nc_dict, k)
        elif order == 2:
            bi = h + (wtok,)
            c = counts[2].get(bi, 0)
            if c > 0:
                # lower order for bigram is unigram mle
                denom -= mle_prob((wtok,), counts[1], {})

    # avoid division by zero
    if denom <= 0:
        return 1.0
    # alpha is the ratio of remaining mass in numerator to remaining mass in denominator
    alpha = numer / denom
    # ensure alpha is non-negative
    if alpha < 0:
        return 1.0
    return alpha

if __name__ == "__main__":
    train_path = Path(TRAIN_FILENAME)
    tokens = list(stream_tokens(train_path))
    counts = count_ngrams(tokens, max_n=4)
    Nc_dict = {n: get_Nc(counts[n]) for n in range(1, 5)}

    # Pick a sample quadgram from training
    quadgrams = list(counts[4].keys())
    if quadgrams:
        h = quadgrams[0][:3]
        w = quadgrams[0][3]
        print("Performing Katz Backoff Probability Calculation for", h, w)
        pk = pkatz(w, h, counts, Nc_dict, k)
        print(f"P_Katz({w} | {h}) = {pk:.12f}")
    else:
        print("No quadgrams found in the data.")


Performing Katz Backoff Probability Calculation for ('Mahendra', 'Singh', 'Dhoni') is
P_Katz(is | ('Mahendra', 'Singh', 'Dhoni')) = 0.013080087919


In [8]:
# kneser-ney smoothing
# p(Wi | Wi-1, Wi-2, Wi-3) = max(c(Wi-3, Wi-2, Wi-1, Wi) - d, 0) / c(Wi-3, Wi-2, Wi-1) + lambda(Wi-3, Wi-2, Wi-1) * p(Wi)
# lambda(Wi-1, Wi-2, Wi-3) = (d / c(Wi-3, Wi-2, Wi-1)) * unique quadrigrams starting with Wi-3, Wi-2, Wi-1 
# P(Wi) = no. of unigram ending with Wi / total no. of unique quadrigrams
"""
Kneser-Ney smoothing implementation for n=1..4 (quadrigram model).
Formulas:
 p_kn(w|h) = max(c(h w) - d, 0)/c(h) + lambda(h) * p_kn_lower(w|h')
 lambda(h) = (d / c(h)) * |{w: c(h w) > 0}|
 Base unigram p_kn_1(w) = continuation_count(w) / total_continuation
"""
from pathlib import Path
from collections import defaultdict, deque, Counter

TRAIN_FILENAME = "msd.txt"
MAX_N = 4
DISCOUNT = 0.75  

def stream_tokens(path: Path):
	with path.open("r", encoding="utf-8", errors="ignore") as f:
		for line in f:
			for tok in line.strip().split():
				t = tok.strip()
				if t:
					yield t

def count_ngrams(tokens, max_n=4):
	counts = {n: defaultdict(int) for n in range(1, max_n + 1)}
	window = deque(maxlen=max_n - 1)
	for tok in tokens:
		counts[1][(tok,)] += 1
		if max_n > 1:
			hist = list(window)
			hl = len(hist)
			for n in range(2, max_n + 1):
				need = n - 1
				if hl >= need:
					gram = tuple(hist[-need:] + [tok])
					counts[n][gram] += 1
		window.append(tok)
	return counts


def compute_continuation_counts(counts):
	# continuation counts for unigram: number of unique left contexts for w (from bigrams)
	cont_uni = Counter()
	for (u, v), c in counts[2].items():
		cont_uni[v] += 1

	# For higher orders, unique continuation counts per history h: |{w: c(h w) > 0}|
	unique_cont = {n: {} for n in range(2, MAX_N + 1)}
	for n in range(2, MAX_N + 1):
		uniq = defaultdict(int)
		for gram in counts[n].keys():
			h = gram[:-1]
			uniq[h] += 1
		unique_cont[n] = uniq

	total_bigram_types = len(counts[2])
	return cont_uni, total_bigram_types, unique_cont


def p_kn_recursive(w, h, counts, cont_uni, total_bigram_types, unique_cont, d=DISCOUNT):
	n = len(h) + 1
	if n == 1:
		# unigram base: continuation probability
		return cont_uni.get(w, 0) / total_bigram_types if total_bigram_types > 0 else 0.0

	# c(h) and c(h w)
	c_h = counts[n - 1].get(h, 0)
	gram = h + (w,)
	c_hw = counts[n].get(gram, 0)

	# first term
	term1 = 0.0
	if c_h > 0:
		term1 = max(c_hw - d, 0.0) / c_h

	# lambda(h)
	lambda_h = 0.0
	if c_h > 0:
		N_cont_h = unique_cont[n].get(h, 0)
		lambda_h = (d / c_h) * N_cont_h

	# lower-order history: drop leftmost word
	lower_h = h[1:]
	p_lower = p_kn_recursive(w, lower_h, counts, cont_uni, total_bigram_types, unique_cont, d)
	return term1 + lambda_h * p_lower

train_path = Path(TRAIN_FILENAME)
tokens = list(stream_tokens(train_path))
counts = count_ngrams(tokens, max_n=MAX_N)
cont_uni, total_bigram_types, unique_cont = compute_continuation_counts(counts)

quad_counts = counts[4]
top_quads = sorted(quad_counts.items(), key=lambda kv: -kv[1])[:10]
print("Top 10 quadrigrams with Kneser-Ney probabilities:")
for gram, c in top_quads:
    h = gram[:-1]
    w = gram[-1]
    p = p_kn_recursive(w, h, counts, cont_uni, total_bigram_types, unique_cont)
    print(f"{' '.join(gram):<40} count={c:<6} p_kn={p:.4f}")



Top 10 quadrigrams with Kneser-Ney probabilities:
at an average of                         count=10     p_kn=0.9971
one of the most                          count=4      p_kn=0.4900
the captain of the                       count=4      p_kn=0.8440
led India to victory                     count=4      p_kn=0.9738
India to victory in                      count=4      p_kn=0.8738
to victory in the                        count=4      p_kn=0.9520
innings at an average                    count=4      p_kn=0.9903
as the captain of                        count=4      p_kn=0.8948
of the most prolific                     count=3      p_kn=0.6659
the Indian cricket team                  count=3      p_kn=0.9558


In [14]:
from pathlib import Path
from collections import Counter, defaultdict, deque
import math
import heapq

TRAIN = Path("msd.txt")
OUT_DIR = Path(".")
MAX_N = 4
BEAM_SIZE = 20
BEAM_CANDIDATES = 50
MAX_LEN = 40

def build_counts(path, max_n=4):
    ngram_counts = {n: Counter() for n in range(1, max_n+1)}
    context_counts = {n: Counter() for n in range(2, max_n+1)}
    vocab = set()

    with open(path, encoding="utf-8") as f:
        for line in f:
            toks = line.strip().split()
            if not toks:
                continue

            sent = ["<s>"] * (max_n-1) + toks + ["</s>"]

            for n in range(1, max_n+1):
                for i in range(len(sent)-n+1):
                    gram = tuple(sent[i:i+n])
                    ngram_counts[n][gram] += 1
                    if n >= 2:
                        ctx = tuple(sent[i:i+n-1])
                        context_counts[n][ctx] += 1

            vocab.update(toks)

    vocab.update({"<s>", "</s>"})
    return ngram_counts, context_counts, sorted(vocab)

ngram_counts, context_counts, VOCAB = build_counts(TRAIN, MAX_N)
TOTAL_UNIGRAMS = sum(ngram_counts[1].values())

def mle_prob(next_word, context):
    """
    Highest-order MLE with backoff.
    context is a tuple of tokens.
    """

    # max usable context = n-1 tokens
    for order in range(min(len(context), MAX_N-1), -1, -1):

        if order == 0:
            # unigram
            count = ngram_counts[1].get((next_word,), 0)
            return count / TOTAL_UNIGRAMS if TOTAL_UNIGRAMS else 0.0

        ctx = tuple(context[-order:])
        denom = context_counts[order+1].get(ctx, 0)
        if denom == 0:
            continue

        num = ngram_counts[order+1].get(ctx + (next_word,), 0)
        return num / denom

    return 0.0

def top_k_candidates(context, k=50):
    heap = []

    for w in VOCAB:
        if w == "<s>":     # do not generate <s> again
            continue

        p = mle_prob(w, context)
        if p <= 0:
            continue

        if len(heap) < k:
            heapq.heappush(heap, (p, w))
        else:
            if p > heap[0][0]:
                heapq.heapreplace(heap, (p, w))

    # heap holds (prob, word) → return sorted descending
    return sorted([(w, p) for p, w in heap], key=lambda x: -x[1])

def generate_greedy(n):
    ctx = deque(["<s>"] * (n-1), maxlen=n-1)
    out = []

    for _ in range(MAX_LEN):
        best_w, best_p = None, -1

        for w in VOCAB:
            if w == "<s>":
                continue

            p = mle_prob(w, tuple(ctx))
            if p > best_p:
                best_p = p
                best_w = w

        if best_w is None:
            break

        out.append(best_w)
        if best_w == "</s>":
            break

        ctx.append(best_w)

    if out and out[-1] == "</s>":
        out = out[:-1]

    return " ".join(out)

def generate_beam_unigram():
    # Beam search for unigram model = pick top tokens by unigram freq
    top_words = ngram_counts[1].most_common(BEAM_SIZE)
    # remove </s> and <s>
    top_words = [w for w, c in top_words if w not in ("<s>", "</s>")]
    if not top_words:
        return ""
    return " ".join(top_words[1])

def generate_beam(n, beam_size=BEAM_SIZE):
    if n == 1:
        return generate_beam_unigram()
    # beam entry = (logP, tokens, context, finished_flag)
    start_ctx = deque(["<s>"] * (n-1), maxlen=n-1)
    beams = [(0.0, [], start_ctx.copy(), False)]
    completed = []

    for _ in range(MAX_LEN):

        new_beams = []

        for logp, toks, ctx, done in beams:

            if done:
                new_beams.append((logp, toks, ctx, True))
                continue

            # expand
            candidates = top_k_candidates(tuple(ctx), BEAM_CANDIDATES)
            if not candidates:
                new_beams.append((logp, toks, ctx, True))
                continue

            for w, p in candidates:
                new_ctx = ctx.copy()
                new_ctx.append(w)
                new_toks = toks + [w]
                new_logp = logp + math.log(p)
                finished = (w == "</s>")
                new_beams.append((new_logp, new_toks, new_ctx, finished))

        if not new_beams:
            break

        # prune to beam_size
        new_beams.sort(key=lambda x: x[0], reverse=True)
        beams = new_beams[:beam_size]

        # collect finished beams
        for lp, tks, c, f in beams:
            if f and tks not in completed:
                completed.append((lp, tks))

        if len(completed) >= beam_size:
            break

    # ranking final sequences
    final_results = completed if completed else [(lp, toks) for lp, toks, _, _ in beams]
    final_results.sort(key=lambda x: x[0], reverse=True)

    # return best sentence
    best = final_results[0][1]
    if best and best[-1] == "</s>":
        best = best[:-1]
    return " ".join(best)

def write_sentences(model_n):
    gfile = OUT_DIR / f"{model_n}gram_greedy.txt"
    bfile = OUT_DIR / f"{model_n}gram_beam.txt"

    with open(gfile, "w", encoding="utf-8") as gf, open(bfile, "w", encoding="utf-8") as bf:
        for _ in range(100):
            gf.write(generate_greedy(model_n) + "\n")
            bf.write(generate_beam(model_n, BEAM_SIZE) + "\n")

for n in range(1, MAX_N+1):
    write_sentences(n)
